# `ToolCallLimitMiddleware`

Middleware that tracks tool-call counts and enforces configurable limits during agent execution.

It can limit all tools together or only one named tool. Limits may apply across an entire conversation thread, within one agent run, or both.

When a limit is exceeded, the middleware can block the extra tool calls, raise an exception, or terminate the agent immediately.

- Bases: `AgentMiddleware[ToolCallLimitState[ResponseT], ContextT, ResponseT]`
- State schema: `ToolCallLimitState`

## Constructor

```python
ToolCallLimitMiddleware(
    *,
    tool_name: str | None = None, # Specific tool to limit; None limits all tools
    thread_limit: int | None = None, # Maximum calls across the thread
    run_limit: int | None = None, # Maximum calls within one agent run
    exit_behavior: ExitBehavior = "continue" # Behaviour after exceeding a limit
)
```

## Parameters

* `tool_name` — Name of the specific tool to track.
  * Default: `None`
  * When `None`, all tool calls are counted together.
  * When set, calls to other tools are ignored by this middleware instance.

* `thread_limit` — Maximum number of matching tool calls allowed across the thread.
  * Default: `None`
  * `None` means no thread-level limit.
  * Allowed calls increase the thread counter.
  * Blocked calls do not increase the stored thread counter.

* `run_limit` — Maximum number of matching tool calls allowed during the current run.
  * Default: `None`
  * `None` means no run-level limit.
  * Both allowed calls and blocked attempts are reflected in the final run counter.

* `exit_behavior` — Determines what happens when a limit is exceeded.
  * Default: `"continue"`
  * Supported values: `"continue"`, `"error"`, and `"end"`.

At least one of `thread_limit` or `run_limit` must be configured.

## Validation

The constructor raises `ValueError` when:

```text
thread_limit is None AND run_limit is None
```

It also raises `ValueError` when `exit_behavior` is not one of:

```python
("continue", "error", "end")
```

When both limits are configured, the run limit cannot be larger than the thread limit:

```text
run_limit <= thread_limit
```

For example, this is invalid:

```python
ToolCallLimitMiddleware(
    thread_limit=5,
    run_limit=8
)
```

because the per-run allowance exceeds the total thread allowance.

## Attributes

* `state_schema` — Set to `ToolCallLimitState`.
* `tool_name` — Specific tool name being limited, or `None` for all tools.
* `thread_limit` — Configured thread-level limit.
* `run_limit` — Configured run-level limit.
* `exit_behavior` — Configured limit-exceeded strategy.

# `ExitBehavior`

Type alias defining the supported limit-exceeded strategies.

```python
ExitBehavior = Literal[
    "continue",
    "error",
    "end"
]
```

## `"continue"`

Blocks only the calls that exceed the limit.

For every blocked call, the middleware creates an error `ToolMessage`. Other allowed tool calls and later model execution can continue.

This is the default behaviour.

## `"error"`

Raises `ToolCallLimitExceededError` as soon as one or more calls exceed the configured limit.

## `"end"`

Injects error `ToolMessage` objects for blocked calls, appends a final `AIMessage` describing the exceeded limits, and returns:

```python
{
    "jump_to": "end"
}
```

This stops agent execution immediately.

When limiting one named tool, `"end"` raises `NotImplementedError` if the same model response also contains pending calls to other tool names, because those calls would otherwise be skipped.

# `ToolCallLimitState`

Agent-state schema used to store tool-call counters.

- Bases: `AgentState[ResponseT]`

```python
class ToolCallLimitState(AgentState[ResponseT]):
    thread_tool_call_count: NotRequired[
        Annotated[
            dict[str, int],
            PrivateStateAttr
        ]
    ]

    run_tool_call_count: NotRequired[
        Annotated[
            dict[str, int],
            UntrackedValue,
            PrivateStateAttr
        ]
    ]
```

## Fields

### `thread_tool_call_count`

Dictionary containing thread-level call counts.

```python
dict[str, int]
```

It is private middleware state but is not marked with `UntrackedValue`, allowing it to participate in persistent thread state.

### `run_tool_call_count`

Dictionary containing call counts for the current run.

```python
dict[str, int]
```

It is marked with:

```python
UntrackedValue
```

so it is not persisted like ordinary tracked graph state.

## Counter Keys

When limiting a specific tool, its name is used as the dictionary key:

```python
{
    "search": 4
}
```

When limiting all tools globally, the special key is:

```python
"__all__"
```

Example:

```python
{
    "__all__": 7
}
```

Using dictionaries allows multiple middleware instances to maintain independent counters.

# `ToolCallLimitExceededError`

Exception raised when a tool-call limit is exceeded and:

```python
exit_behavior="error"
```

- Bases: `Exception`

## Constructor

```python
ToolCallLimitExceededError(
    thread_count: int, # Current or hypothetical thread count
    run_count: int, # Current run count
    thread_limit: int | None, # Configured thread limit
    run_limit: int | None, # Configured run limit
    tool_name: str | None = None # Limited tool, or None for all tools
)
```

## Attributes

* `thread_count` — Count used to determine whether the thread limit was exceeded.
* `run_count` — Count used to determine whether the run limit was exceeded.
* `thread_limit` — Configured thread-level limit.
* `run_limit` — Configured run-level limit.
* `tool_name` — Limited tool name, or `None` when all tools are limited.

The exception message is generated using `_build_final_ai_message_content`.

Example:

```text
'search' tool call limit reached: thread limit exceeded (6/5 calls).
```

# Properties

## `name`

Returns the middleware instance name.

```python
@property
def name(
    self
) -> str
```

When no specific tool is configured:

```text
ToolCallLimitMiddleware
```

When `tool_name="search"`:

```text
ToolCallLimitMiddleware[search]
```

This allows multiple middleware instances targeting different tools to have distinct names.

# Methods

## 1. `_would_exceed_limit`

Checks whether one additional matching tool call would exceed either configured limit.

```python
_would_exceed_limit(
    self,
    thread_count: int,
    run_count: int
) -> bool
```

The logic is equivalent to:

```python
(
    thread_limit is not None
    and thread_count + 1 > thread_limit
)
or
(
    run_limit is not None
    and run_count + 1 > run_limit
)
```

A call is allowed when incrementing both relevant counters remains within the configured limits.

## 2. `_matches_tool_filter`

Checks whether one tool call should be tracked by this middleware instance.

```python
_matches_tool_filter(
    self,
    tool_call: ToolCall
) -> bool
```

Returns `True` when:

```python
self.tool_name is None
```

or:

```python
tool_call["name"] == self.tool_name
```

## 3. `_separate_tool_calls`

Separates matching tool calls into allowed and blocked groups.

```python
_separate_tool_calls(
    self,
    tool_calls: list[ToolCall],
    thread_count: int,
    run_count: int
) -> tuple[
    list[ToolCall], # Allowed matching calls
    list[ToolCall], # Blocked matching calls
    int, # Final allowed thread count
    int # Final allowed run count
]
```

Tool calls are evaluated in their original order.

For each matching call:

1. Check whether one more call would exceed a configured limit.
2. Add it to `blocked_calls` when the limit would be exceeded.
3. Otherwise add it to `allowed_calls`.
4. Increment the temporary thread and run counters only for allowed calls.

Calls that do not match `tool_name` are ignored by this method.

## 4. `after_model`

Reads tool calls from the latest `AIMessage`, updates counters, and enforces the configured limits.

```python
@hook_config(can_jump_to=["end"])
def after_model(
    self,
    state: ToolCallLimitState[ResponseT],
    runtime: Runtime[ContextT]
) -> dict[str, Any] | None
```

### Behaviour

The method:

1. Reads `state["messages"]`.
2. Searches backward for the latest `AIMessage`.
3. Returns `None` when no tool calls exist.
4. Loads the current thread and run counters.
5. Separates matching calls into allowed and blocked groups.
6. Updates both counter dictionaries.
7. Applies the configured `exit_behavior`.

### No Blocked Calls

When matching calls are allowed, it returns updated counters:

```python
{
    "thread_tool_call_count": thread_counts,
    "run_tool_call_count": run_counts,
}
```

When no matching tool calls are present, it returns `None`.

### Blocked Calls

For each blocked call, it creates:

```python
ToolMessage(
    content=<limit message>,
    tool_call_id=tool_call["id"],
    name=tool_call.get("name"),
    status="error",
)
```

The exact returned update depends on `exit_behavior`.

## 5. `aafter_model`

Asynchronous version of `after_model`.

```python
@hook_config(can_jump_to=["end"])
async def aafter_model(
    self,
    state: ToolCallLimitState[ResponseT],
    runtime: Runtime[ContextT]
) -> dict[str, Any] | None
```

It delegates directly to the synchronous implementation:

```python
return self.after_model(state, runtime)
```

# Counting Behaviour

Suppose the current counters are:

```text
thread count = 2
run count    = 2
thread limit = 3
run limit    = 3
```

and the next AI response requests two matching tool calls.

The calls are evaluated sequentially:

```text
Call 1 -> allowed
Call 2 -> blocked
```

The stored counts become:

```text
thread count = 3
run count    = 4
```

The distinction is intentional:

* The thread counter includes only calls allowed to proceed.
* The run counter includes allowed calls plus blocked attempts produced during that run.

# Limit Evaluation Order

Tool calls are processed in the order returned by the model.

Example:

```python
tool_calls = [
    {"name": "search", "id": "1", "args": {}},
    {"name": "search", "id": "2", "args": {}},
    {"name": "search", "id": "3", "args": {}},
]
```

With:

```text
current run count = 1
run limit         = 2
```

the result is:

```text
Call 1 -> allowed
Call 2 -> blocked
Call 3 -> blocked
```

Only the first available call fits within the remaining allowance.

# Exit Behaviour Details

## Continue

```python
ToolCallLimitMiddleware(
    run_limit=3,
    exit_behavior="continue"
)
```

Blocked calls receive error messages such as:

```text
Tool call limit exceeded. Do not make additional tool calls.
```

When limiting one named tool:

```text
Tool call limit exceeded. Do not call 'search' again.
```

The returned update contains:

```python
{
    "thread_tool_call_count": thread_counts,
    "run_tool_call_count": run_counts,
    "messages": artificial_tool_messages,
}
```

Allowed calls to other tools remain available for execution.

## Error

```python
ToolCallLimitMiddleware(
    tool_name="search",
    thread_limit=5,
    exit_behavior="error"
)
```

When the next call would exceed the limit, the middleware raises:

```python
ToolCallLimitExceededError
```

For error reporting, the middleware uses a hypothetical thread count that includes the blocked calls. This allows the error message to show the count that would have exceeded the limit.

## End

```python
ToolCallLimitMiddleware(
    run_limit=5,
    exit_behavior="end"
)
```

The middleware returns:

```python
{
    "thread_tool_call_count": thread_counts,
    "run_tool_call_count": run_counts,
    "jump_to": "end",
    "messages": [
        *blocked_tool_messages,
        AIMessage(content=<final limit description>),
    ],
}
```

The final message may look like:

```text
Tool call limit reached: run limit exceeded (6/5 calls).
```

For a named tool:

```text
'search' tool call limit reached: thread limit exceeded (6/5 calls).
```

## Parallel Calls with `"end"`

When a middleware limits one named tool and the same model response also requests other tools, immediate termination is unsupported.

Example:

```text
search    -> exceeds limit
calculator -> still pending
```

The middleware raises:

```python
NotImplementedError
```

with a message recommending:

```text
Use 'continue' or 'error' behavior instead.
```

At this pinned revision, the explicit guard checks for pending calls to other tool names when `tool_name` is configured.

# Helper Functions

## `_build_tool_message_content`

Builds the concise error text sent back to the model for a blocked call.

```python
_build_tool_message_content(
    tool_name: str | None
) -> str
```

For one named tool:

```text
Tool call limit exceeded. Do not call '<tool_name>' again.
```

For all tools:

```text
Tool call limit exceeded. Do not make additional tool calls.
```

The message intentionally avoids thread/run terminology because the model does not need the middleware's internal counting concepts.

## `_build_final_ai_message_content`

Builds the detailed user-facing limit message.

```python
_build_final_ai_message_content(
    thread_count: int,
    run_count: int,
    thread_limit: int | None,
    run_limit: int | None,
    tool_name: str | None
) -> str
```

It reports each exceeded limit.

Thread example:

```text
Tool call limit reached: thread limit exceeded (11/10 calls).
```

Run example:

```text
Tool call limit reached: run limit exceeded (6/5 calls).
```

When both limits are exceeded:

```text
Tool call limit reached: thread limit exceeded (...) and run limit exceeded (...).
```

# Multiple Middleware Instances

Separate instances can limit different tools independently.

```python
middleware = [
    ToolCallLimitMiddleware(
        tool_name="search",
        run_limit=3,
    ),
    ToolCallLimitMiddleware(
        tool_name="send_email",
        thread_limit=2,
    ),
]
```

The state dictionaries may contain:

```python
{
    "search": 3,
    "send_email": 1,
}
```

A global instance can also use the special `__all__` key:

```python
ToolCallLimitMiddleware(
    run_limit=10
)
```

# Examples

## Limit All Tool Calls Per Run

```python
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware

agent = create_agent(
    model="openai:gpt-5.5",
    tools=[search, calculator, send_email],
    middleware=[
        ToolCallLimitMiddleware(
            run_limit=5
        )
    ],
)
```

After five allowed calls, additional calls are blocked and returned to the model as error tool results.

## Apply Thread and Run Limits

```python
middleware = ToolCallLimitMiddleware(
    thread_limit=20,
    run_limit=8,
    exit_behavior="continue",
)
```

This allows at most:

```text
20 calls across the thread
8 calls during one run
```

## Limit One Tool

```python
middleware = ToolCallLimitMiddleware(
    tool_name="search",
    thread_limit=10,
    run_limit=4,
)
```

Only calls whose name is exactly `"search"` are counted.

## Raise an Exception

```python
from langchain.agents.middleware import (
    ToolCallLimitExceededError,
    ToolCallLimitMiddleware,
)

middleware = ToolCallLimitMiddleware(
    tool_name="search",
    thread_limit=5,
    exit_behavior="error",
)

try:
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Research this topic.",
                }
            ]
        }
    )
except ToolCallLimitExceededError as exc:
    print(exc.thread_count)
    print(exc.thread_limit)
    print(exc)
```

## Stop Agent Execution

```python
middleware = ToolCallLimitMiddleware(
    run_limit=5,
    exit_behavior="end",
)
```

The middleware appends a final limit message and jumps directly to the agent's end node.

# Source

This reference follows the pinned LangChain source:

```text
libs/langchain_v1/langchain/agents/middleware/tool_call_limit.py
Commit: 42f8f79293cfb7589e5bc1d74a8ae4dfd0bf15e3
```